# Generate Li-Deficient Ta-Doped LLZTO Structures

This notebook documents the structure-generation step used to construct Li-deficient, Ta-doped LLZO garnet structures from a CIF template.

The structure-generation logic follows three main steps:

1. identify Li sublattices from the CIF-derived template structure;
2. introduce Li vacancies according to target Li-site occupancies;
3. substitute selected Zr atoms with Ta dopants.

The generated structures are written in VASP POSCAR format and can be used as starting configurations for subsequent relaxation, molecular dynamics, phonon, and Raman calculations.

This notebook is included to document the structure-preparation logic. The processed phonon and Raman datasets used in the later notebooks were generated from re-relaxed structures sampled from molecular dynamics trajectories.

In [1]:
from pathlib import Path
import itertools
import random

import numpy as np
from ase import io

In [2]:
PROJECT_ROOT = Path.cwd().parent

INPUT_CIF = PROJECT_ROOT / "data" / "input_structures" / "awaka_small.cif"
OUTPUT_DIR = PROJECT_ROOT / "data" / "generated_structures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Input CIF: {INPUT_CIF}")
print(f"Output directory: {OUTPUT_DIR}")

Input CIF: c:\Users\shafn\OneDrive\Documents\GitHub\llzto-first-principles-raman\data\input_structures\awaka_small.cif
Output directory: c:\Users\shafn\OneDrive\Documents\GitHub\llzto-first-principles-raman\data\generated_structures


In [3]:
template = io.read(INPUT_CIF, index=0)
symbols = np.array(template.get_chemical_symbols())

print(f"Number of atoms in template: {len(template)}")
print(f"Template formula: {template.get_chemical_formula()}")

unique, counts = np.unique(symbols, return_counts=True)
composition_table = dict(zip(unique, counts))
composition_table

Number of atoms in template: 128
Template formula: La12Li60O48Zr8


{'La': 12, 'Li': 60, 'O': 48, 'Zr': 8}

## Li-site identification logic

The CIF-derived template contains Li atoms on partially occupied crystallographic sites. The original structure-generation code distinguishes two Li-site types using local geometric criteria:

- Li2 pairs are identified as pairs of Li atoms separated by less than 1 Å.
- Li1 sites are the remaining Li atoms that are not part of these close Li2 pairs.
- Local Li1–Li2–Li2–Li1 clusters are then identified using Li1–Li2 distances around 1.58–1.61 Å.

These geometric thresholds are specific to the input CIF template and should be interpreted as part of the structure-construction heuristic used in the original project.

In [4]:
def identify_li_sites_and_clusters(atoms):
    """Identify Li1 sites, Li2 paired sites, and Li1-Li2-Li2-Li1 clusters."""
    symbols = np.array(atoms.get_chemical_symbols())

    li2_pair_indices = []

    for i in range(len(symbols)):
        for j in range(i + 1, len(symbols)):
            if symbols[i] == "Li" and symbols[j] == "Li":
                distance = atoms.get_distance(i, j, mic=True)
                if distance < 1.0:
                    li2_pair_indices.append((i, j))

    li2_flat = set(np.array(li2_pair_indices).flatten())
    li1_indices = [i for i, symbol in enumerate(symbols) if symbol == "Li" and i not in li2_flat]

    li_clusters = []

    for li2_1, li2_2 in li2_pair_indices:
        li1_1 = None
        li1_2 = None

        for i in range(len(symbols)):
            distance = atoms.get_distance(li2_1, i, mic=True)
            if 1.58 < distance < 1.61:
                if i in li1_indices:
                    li1_1 = i
                    break

        for i in range(len(symbols)):
            distance = atoms.get_distance(li2_2, i, mic=True)
            if 1.58 < distance < 1.61:
                if i in li1_indices:
                    li1_2 = i
                    break

        if li1_1 is not None and li1_2 is not None:
            li_clusters.append((li1_1, li2_1, li2_2, li1_2))

    return li1_indices, li2_pair_indices, li_clusters

In [5]:
li1_indices, li2_pair_indices, li_clusters = identify_li_sites_and_clusters(template)

print(f"Number of Li1 sites: {len(li1_indices)}")
print(f"Number of Li2 pairs: {len(li2_pair_indices)}")
print(f"Number of Li2 sites: {2 * len(li2_pair_indices)}")
print(f"Number of Li1-Li2-Li2-Li1 clusters: {len(li_clusters)}")

Number of Li1 sites: 12
Number of Li2 pairs: 24
Number of Li2 sites: 48
Number of Li1-Li2-Li2-Li1 clusters: 24


In [6]:
def generate_structure(
    atoms,
    li1_occupancy,
    li2_occupancy,
    ta_doping=0.25,
    seed=42,
):
    """Generate one Li-deficient, Ta-doped LLZTO structure.

    Parameters
    ----------
    atoms : ase.Atoms
        CIF-derived template structure.
    li1_occupancy : float
        Target fractional occupancy of Li1 sites.
    li2_occupancy : float
        Target fractional occupancy of Li2 sites.
    ta_doping : float
        Fraction of Zr sites replaced by Ta.
    seed : int
        Random seed for reproducible Li vacancy and Ta substitution choices.
    """
    if not (0 <= li1_occupancy <= 1):
        raise ValueError("li1_occupancy must be between 0 and 1.")
    if not (0 <= li2_occupancy <= 0.5):
        raise ValueError("li2_occupancy must be between 0 and 0.5.")
    if not (0 <= ta_doping <= 1):
        raise ValueError("ta_doping must be between 0 and 1.")

    rng = random.Random(seed)

    symbols = np.array(atoms.get_chemical_symbols())
    li1_indices, li2_pair_indices, li_clusters = identify_li_sites_and_clusters(atoms)

    n_li1_sites = len(li1_indices)
    n_li2_sites = 2 * len(li2_pair_indices)

    n_li1_to_delete = int(np.round(n_li1_sites * (1 - li1_occupancy)))
    n_li2_to_keep = int(np.round(n_li2_sites * li2_occupancy))

    # Choose one Li1 vacancy configuration reproducibly.
    li1_delete_options = list(itertools.combinations(li1_indices, n_li1_to_delete))
    li1_indices_to_delete = rng.choice(li1_delete_options) if li1_delete_options else tuple()

    occupancies = {i: True for i in li1_indices}
    for i in li1_indices_to_delete:
        occupancies[i] = False

    double_vacancy = []
    single_vacancy = []
    no_vacancy = []

    for li1_1, li2_1, li2_2, li1_2 in li_clusters:
        if occupancies[li1_1] and occupancies[li1_2]:
            no_vacancy.append((li1_1, li2_1, li2_2, li1_2))
        elif occupancies[li1_1] or occupancies[li1_2]:
            single_vacancy.append((li1_1, li2_1, li2_2, li1_2))
        else:
            double_vacancy.append((li1_1, li2_1, li2_2, li1_2))

    li2_occupied = []

    def fill_clusters(valid_clusters, n_to_fill):
        if n_to_fill <= 0:
            return

        clusters_to_fill = rng.sample(valid_clusters, min(n_to_fill, len(valid_clusters)))

        for cluster in clusters_to_fill:
            if occupancies[cluster[0]] and not occupancies[cluster[3]]:
                li2_occupied.append(cluster[2])
            elif occupancies[cluster[3]] and not occupancies[cluster[0]]:
                li2_occupied.append(cluster[1])
            else:
                li2_occupied.append(cluster[2] if rng.random() >= 0.5 else cluster[1])

    # Prefer filling Li2 sites near Li1 vacancies.
    priority_clusters = double_vacancy + single_vacancy
    n_priority = min(n_li2_to_keep, len(priority_clusters))
    fill_clusters(priority_clusters, n_priority)

    remaining = n_li2_to_keep - len(li2_occupied)
    fill_clusters(no_vacancy, remaining)

    all_li2_indices = list(np.array(li2_pair_indices).flatten())
    li2_indices_to_delete = [i for i in all_li2_indices if i not in li2_occupied]

    result = atoms.copy()

    # Replace selected Zr atoms with Ta.
    zr_indices = [i for i, symbol in enumerate(symbols) if symbol == "Zr"]
    n_ta = int(np.round(len(zr_indices) * ta_doping))
    ta_indices = rng.sample(zr_indices, n_ta)

    for i in ta_indices:
        result[i].symbol = "Ta"

    # Delete Li atoms only after all substitutions are complete, so original indices remain valid.
    indices_to_delete = sorted(list(li1_indices_to_delete) + li2_indices_to_delete, reverse=True)

    for i in indices_to_delete:
        del result[i]

    metadata = {
        "n_li1_sites": n_li1_sites,
        "n_li2_sites": n_li2_sites,
        "n_li1_deleted": len(li1_indices_to_delete),
        "n_li2_kept": len(li2_occupied),
        "n_ta_substituted": len(ta_indices),
        "li1_indices_deleted": list(li1_indices_to_delete),
        "li2_indices_kept": list(li2_occupied),
        "ta_indices": list(ta_indices),
        "formula": result.get_chemical_formula(),
    }

    return result, metadata

In [7]:
generated_structure, metadata = generate_structure(
    template,
    li1_occupancy=0.91,
    li2_occupancy=0.15,
    ta_doping=0.25,
    seed=RANDOM_SEED,
)

metadata

{'n_li1_sites': 12,
 'n_li2_sites': 48,
 'n_li1_deleted': 1,
 'n_li2_kept': 7,
 'n_ta_substituted': 2,
 'li1_indices_deleted': [10],
 'li2_indices_kept': [17, 36, 19, 38, 42, 50, 15],
 'ta_indices': [78, 72],
 'formula': 'La12Li18O48Ta2Zr6'}

In [9]:
output_file = OUTPUT_DIR / "generated_Li4p5_example.vasp"

io.write(
    output_file,
    generated_structure,
    format="vasp",
    sort=True,
    direct=True,
    vasp5=True,
)

print(f"Generated formula: {generated_structure.get_chemical_formula()}")
print(f"Saved structure to: {output_file}")

Generated formula: La12Li18O48Ta2Zr6
Saved structure to: c:\Users\shafn\OneDrive\Documents\GitHub\llzto-first-principles-raman\data\generated_structures\generated_Li4p5_example.vasp
